# 🛡️ ZENTRA — Person Detection Training (Kaggle, ฟรี 30 ชม./สัปดาห์)

เทรน **person detector** — รากฐานของทั้งระบบ (PPE / Zone / Fall ยืนบนกล่องคนทั้งหมด)

> **ทำไมย้ายจาก Colab มา Kaggle:** Colab free ให้ GPU แค่ ~3-4 ชม./วัน (เทรนรอบก่อนโดนตัดที่ epoch 81 งานหายหมด)
> Kaggle ให้ **30 ชม./สัปดาห์** และมี **"Save & Run All" ที่รันแบบ background** → ปิดคอมได้ เน็ตหลุดได้ งานไม่หาย

---
## ⚙️ ตั้งค่าก่อนรัน (สำคัญมาก — ทำ 2 อย่างนี้ก่อน)

แถบขวามือ (Session options):
1. **Accelerator → `GPU T4 x2`** (ถ้าไม่เปิด จะเทรนบน CPU ช้ามาก)
2. **Internet → `On`** (ต้องใช้โหลด ultralytics + dataset — Kaggle อาจให้ยืนยันเบอร์โทรก่อน)

## ▶️ วิธีรัน (แนะนำ)

กดปุ่ม **"Save Version" → เลือก "Save & Run All (Commit)"** มุมขวาบน
→ Kaggle จะรันทุกเซลล์บนเซิร์ฟเวอร์ของมันเอง **คุณปิดเบราว์เซอร์ได้เลย** กลับมาดูผลทีหลัง

(หรือจะกด Run ทีละเซลล์ก็ได้ แต่ต้องเปิดแท็บทิ้งไว้)

## 🎯 เป้าหมาย

รอบก่อน (40 epochs) ได้ `test mAP50 = 0.816 · recall = 0.774` และ**กราฟยังไต่ขึ้นอยู่ = ยังเทรนไม่จบ**
รอบนี้เทรน **100 epochs** → คาดว่า recall จะขึ้นอีก (เป้าระบบความปลอดภัย: **recall ≥ 0.90**)

## 1) เช็ค GPU + ติดตั้ง

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
print('GPU            :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'ยังไม่เปิด GPU → แถบขวา Accelerator → GPU T4 x2'

!pip -q install ultralytics roboflow
import ultralytics; print('ultralytics', ultralytics.__version__)

## 2) โหลด dataset จาก Roboflow

ใช้ dataset ที่ fork ไว้แล้ว: `krittpass-workspace/people-detection-o4rdr-scnrf-m10zy` **v2**
(7,180 ภาพ · 1 คลาส `person` · ภาพมุมกล้อง CCTV ในร่ม — โดเมนใกล้เคียงกล้องโรงงาน)

> 🔐 **API key:** ทางที่ปลอดภัยคือใช้ **Kaggle Secrets** — แถบขวา **Add-ons → Secrets → Add secret**
> ตั้งชื่อ `ROBOFLOW_API_KEY` แล้วใส่ค่า key (จะได้ไม่ติดไปกับ notebook ตอนแชร์)
> ถ้าไม่ตั้ง Secret เซลล์จะถามให้วาง key ตรง ๆ แทน
>
> ⚠️ key เดิมของคุณเคยถูกเผยในแชต — **ควรไป Regenerate ที่ roboflow.com → Settings ก่อน** แล้วใช้ตัวใหม่

In [ ]:
# ── API key: ลองอ่านจาก Kaggle Secrets ก่อน, ไม่มีค่อย fallback ──
API_KEY = ''
try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
    print('✅ อ่าน API key จาก Kaggle Secrets แล้ว')
except Exception:
    API_KEY = 'วาง_ROBOFLOW_API_KEY_ตรงนี้'   # ← ถ้าไม่ใช้ Secrets ให้วางที่นี่
    print('⚠️ ไม่พบ Secret — ใช้ key ที่วางไว้ในเซลล์')

assert API_KEY and 'วาง_' not in API_KEY, 'ยังไม่ได้ใส่ API key'

from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)
project = rf.workspace('krittpass-workspace').project('people-detection-o4rdr-scnrf-m10zy')
dataset = project.version(2).download('yolov11', location='/kaggle/working/dataset')

DATA_YAML = dataset.location + '/data.yaml'
print('✅ DATA_YAML =', DATA_YAML)

## 3) เทรน — 100 epochs @ imgsz 640

**ทำไม 640 ไม่ใช่ 960:** เครื่อง deploy จริง (โน้ตบุ๊ก CPU) ตั้ง `PERSON_IMGSZ=640` ใน `backend/.env`
เพราะ 960 ได้แค่ ~3.3 fps → **เทรนที่ res เดียวกับที่ใช้จริง** ไม่เสียเวลาเทรนที่ res ที่ไม่ได้ใช้

**ทำไม 100 epochs:** กราฟรอบก่อนที่ epoch 40 — loss ยังลง, mAP/recall ยังขึ้น = **ยังไม่อิ่ม**
ตั้ง `patience=25` ให้มันหยุดเองเมื่ออิ่มจริง

> เซฟลง `/kaggle/working/runs` → weights จะอยู่ใน **Output ของ notebook** ดาวน์โหลดได้แม้ session จบ
> (บทเรียนจากรอบก่อน: เซฟไว้ที่ temp → runtime ตาย → งาน 1.5 ชม. หายหมด)

⏱️ ใช้เวลา ~1.5-2 ชม. บน T4

In [ ]:
from ultralytics import YOLO
import time

model = YOLO('yolo11s.pt')
t0 = time.time()
model.train(
    data      = DATA_YAML,
    epochs    = 100,          # รอบก่อน 40 ยังไม่อิ่ม (กราฟยังไต่ขึ้น)
    imgsz     = 640,          # ตรงกับ PERSON_IMGSZ ที่ deploy จริง
    batch     = 16,
    device    = 0,
    workers   = 4,
    patience  = 25,           # หยุดเองเมื่อไม่ดีขึ้น 25 epochs
    optimizer = 'auto', cos_lr = True, seed = 0, pretrained = True,
    # augmentation (ultralytics ทำสดทุก epoch — Roboflow ไม่ได้อบมา = ไม่มี leakage)
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=0.0, translate=0.1,
    scale=0.5, fliplr=0.5, mosaic=1.0, close_mosaic=15,
    project='/kaggle/working/runs', name='person_v2', exist_ok=True,
    plots=True, val=True,
)
print(f'\n✅ เทรนเสร็จใน {(time.time()-t0)/60:.1f} นาที')
BEST = '/kaggle/working/runs/person_v2/weights/best.pt'
print('best.pt =', BEST)

## 4) วัดผลอย่างซื่อสัตย์

เทียบกับ baseline COCO และกับผลรอบก่อน (40 epochs) เพื่อดูว่า **ดีขึ้นจริงไหม**

| ชุด | รอบก่อน (40 ep) |
|---|---|
| baseline COCO — val | mAP50 0.478 · R 0.434 |
| fine-tuned — val | mAP50 0.570 · R 0.528 |
| fine-tuned — **test** | mAP50 **0.816** · R **0.774** |

> **หมายเหตุ:** val ของ dataset นี้แน่นกว่า test ~2 เท่า (7.5 vs 3.6 คน/ภาพ)
> → val จะได้คะแนนต่ำกว่าเสมอ **ไม่ใช่โมเดลแย่** ให้ดู test เป็นหลัก และดู **recall** สำคัญที่สุด
> (คนที่ตรวจไม่เจอ = ไม่มีตัวตนในระบบ ไม่มี PPE/Zone/Fall)

In [ ]:
from ultralytics import YOLO

ft   = YOLO(BEST)
base = YOLO('yolo11s.pt')      # COCO baseline (person = class 0)

def ev(m, tag, split, **kw):
    r = m.val(data=DATA_YAML, split=split, imgsz=640, device=0,
              verbose=False, plots=False, **kw)
    print(f'{tag:<26} {split:<5}  mAP50={r.box.map50:.4f}  mAP50-95={r.box.map:.4f}  '
          f'P={r.box.mp:.4f}  R={r.box.mr:.4f}')
    return r.box.map50, r.box.mr

print('===== Person Stage 1 — 100 epochs =====')
ev(base, 'baseline COCO yolo11s', 'val', classes=[0])
ev(ft,   'fine-tuned person_v2',  'val')
m50, rec = ev(ft, 'fine-tuned person_v2', 'test')   # แตะ test ครั้งเดียว ตอนจบ

print(f'\n📊 เทียบกับรอบก่อน (40 ep): test mAP50 0.816 → {m50:.4f} | recall 0.774 → {rec:.4f}')
print(f"เป้าระบบความปลอดภัย recall >= 0.90: {'✅ ถึงแล้ว' if rec >= 0.90 else '⬜ ยังไม่ถึง'} ({rec:.4f})")

## 5) เอาโมเดลกลับไปใช้ใน ZENTRA

หลังรันจบ: ไปที่แท็บ **Output** ของ notebook (แถบขวา) → เข้าโฟลเดอร์ `runs/person_v2/weights/`
→ ดาวน์โหลด **`best.pt`**

**วิธีติดตั้งลง ZENTRA:**
1. เปลี่ยนชื่อไฟล์เป็น `person_v1.pt`
2. วางทับที่ `backend/models/person_v1.pt`
3. เปิด `backend/.env` → **เอา `#` ออกหน้าบรรทัด** `PERSON_MODEL=...` (ตอนนี้ผมคอมเมนต์ไว้เพราะโมเดลเก่าเป็นของเล่น)
4. รันแอปใหม่ → ใช้โมเดลที่เทรนแล้วทันที

อย่าลืมดาวน์โหลด `runs/person_v2/results.png` + `val_batch0_pred.jpg` มาดูด้วย — **ภาพสำคัญกว่าตัวเลข**

In [ ]:
import shutil, os

# ก๊อป best.pt ขึ้นมาไว้ชั้นบนสุดของ Output ให้หาง่าย
shutil.copy(BEST, '/kaggle/working/person_v1.pt')

# ลบ dataset ออกจาก output (ไม่งั้น output บวมเป็น GB โดยไม่จำเป็น)
shutil.rmtree('/kaggle/working/dataset', ignore_errors=True)

print('✅ พร้อมดาวน์โหลดจากแท็บ Output:')
for f in sorted(os.listdir('/kaggle/working')):
    print('  -', f)
print('\n📥 person_v1.pt  → วางที่ backend/models/ แล้วเปิดคอมเมนต์ PERSON_MODEL ใน .env')